# Yelp RAG Agent — Benchmark Analysis

Reads results from Colab A100 experiments and produces publication-ready charts.

| Input | Description |
|---|---|
| `results/deployment_perf.json` | VRAM, TTFT, TPS from colab_awq_deploy.ipynb |
| `results/eval_lmdeploy_fp16.csv` | Quality eval — fp16 backend |
| `results/eval_lmdeploy_awq.csv` | Quality eval — AWQ backend |

**Output:** PNG charts saved to `docs/`

## Cell 1 — Setup

In [ ]:
import json, csv, os
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib
import numpy as np

matplotlib.rcParams['font.family'] = 'DejaVu Sans'
matplotlib.rcParams['figure.dpi'] = 150

ROOT     = Path("..")
RESULTS  = ROOT / "results"
DOCS     = ROOT / "docs"
DOCS.mkdir(exist_ok=True)

SCORE_COLS = [
    "score_correctness", "score_evidence",
    "score_groundedness", "score_tool_use", "score_efficiency"
]
SCORE_LABELS = ["Correctness", "Evidence", "Groundedness", "Tool Use", "Efficiency"]
SYSTEMS  = ["direct_llm", "rag_baseline", "full_agent"]
SYS_LABELS = ["Direct LLM", "RAG Baseline", "Full Agent"]
COLORS   = ["#4C72B0", "#DD8452", "#55A868"]

# ── Load deployment perf ──────────────────────────────────────────
with open(RESULTS / "deployment_perf.json", encoding="utf-8") as f:
    perf = json.load(f)
experiments = {e["experiment_id"]: e for e in perf["experiments"]}

# ── Load quality eval CSVs ────────────────────────────────────────
def load_csv(fname):
    with open(RESULTS / fname, encoding="utf-8", errors="replace") as f:
        return list(csv.DictReader(f))

rows_fp16 = load_csv("eval_lmdeploy_fp16.csv")
rows_awq  = load_csv("eval_lmdeploy_awq.csv")

def avg_scores(rows, system):
    sys_rows = [r for r in rows if r["system"] == system]
    result = {}
    for col in SCORE_COLS:
        vals = [float(r[col]) for r in sys_rows if r.get(col, "") != ""]
        result[col] = round(sum(vals) / len(vals), 3) if vals else 0.0
    return result

print("Data loaded.")
print(f"  deployment_perf: {list(experiments.keys())}")
print(f"  fp16 CSV: {len(rows_fp16)} rows")
print(f"  awq  CSV: {len(rows_awq)}  rows")

## Cell 2 — Deployment Performance Charts

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle("Deployment Performance: fp16 vs AWQ  (Colab A100 80GB)",
             fontsize=13, fontweight="bold", y=1.02)

backends   = ["fp16", "AWQ"]
bar_colors = ["#4C72B0", "#DD8452"]
exp_keys   = ["lmdeploy_fp16", "lmdeploy_awq"]

metrics = [
    ("ttft_ms",        "TTFT (ms)",            "Time to First Token"),
    ("estimated_tps",  "Throughput (tokens/s)", "Estimated Throughput"),
    ("avg_response_s", "Avg Response (s)",      "Average Response Time"),
    ("vram_peak_gb",   "VRAM Peak (GB)",        "Peak GPU Memory"),
]

for ax, (key, ylabel, title) in zip(axes, metrics):
    vals = [experiments[ek][key] for ek in exp_keys]
    bars = ax.bar(backends, vals, color=bar_colors, width=0.5, edgecolor="white", linewidth=0.8)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(vals)*0.02,
                f"{val}", ha="center", va="bottom", fontsize=10, fontweight="bold")
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.set_ylabel(ylabel, fontsize=9)
    ax.set_ylim(0, max(vals) * 1.25)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
out = DOCS / "deployment_performance.png"
plt.savefig(out, bbox_inches="tight")
plt.show()
print(f"Saved: {out}")

## Cell 3 — Quality Auto Metrics by System

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle("Quality Eval — Auto Metrics by System  (fp16 backend)",
             fontsize=13, fontweight="bold", y=1.02)

def sys_metric(rows, system, col, cast=float):
    vals = [cast(r[col]) for r in rows if r["system"] == system and r.get(col, "") != ""]
    return round(sum(vals)/len(vals), 2) if vals else 0

# ── Avg elapsed time ──
ax = axes[0]
vals = [sys_metric(rows_fp16, s, "elapsed_seconds") for s in SYSTEMS]
bars = ax.bar(SYS_LABELS, vals, color=COLORS, edgecolor="white")
for bar, v in zip(bars, vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
            f"{v}s", ha="center", fontsize=9, fontweight="bold")
ax.set_title("Avg Response Time (s)", fontweight="bold")
ax.set_ylabel("Seconds")
ax.spines[["top","right"]].set_visible(False)
ax.grid(axis="y", alpha=0.3)

# ── Avg tool count ──
ax = axes[1]
vals = [sys_metric(rows_fp16, s, "tool_count", int) for s in SYSTEMS]
bars = ax.bar(SYS_LABELS, vals, color=COLORS, edgecolor="white")
for bar, v in zip(bars, vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
            f"{v}", ha="center", fontsize=9, fontweight="bold")
ax.set_title("Avg Tool Calls", fontweight="bold")
ax.set_ylabel("Count")
ax.spines[["top","right"]].set_visible(False)
ax.grid(axis="y", alpha=0.3)

# ── Evidence rate ──
ax = axes[2]
def evidence_rate(rows, system):
    sys_rows = [r for r in rows if r["system"]==system]
    return round(sum(1 for r in sys_rows if r["has_evidence"].strip().lower()=="true")/len(sys_rows)*100, 1)
vals = [evidence_rate(rows_fp16, s) for s in SYSTEMS]
bars = ax.bar(SYS_LABELS, vals, color=COLORS, edgecolor="white")
for bar, v in zip(bars, vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
            f"{v}%", ha="center", fontsize=9, fontweight="bold")
ax.set_title("Evidence Rate (%)", fontweight="bold")
ax.set_ylabel("Percentage")
ax.set_ylim(0, 115)
ax.spines[["top","right"]].set_visible(False)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
out = DOCS / "quality_auto_metrics.png"
plt.savefig(out, bbox_inches="tight")
plt.show()
print(f"Saved: {out}")

## Cell 4 — Manual Scores by System (fp16)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

x = np.arange(len(SCORE_LABELS))
width = 0.25

for i, (sys, label, color) in enumerate(zip(SYSTEMS, SYS_LABELS, COLORS)):
    scores = avg_scores(rows_fp16, sys)
    vals   = [scores[c] for c in SCORE_COLS]
    bars   = ax.bar(x + i*width - width, vals, width, label=label,
                    color=color, edgecolor="white", alpha=0.9)
    for bar, v in zip(bars, vals):
        if v > 0:
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.03,
                    f"{v:.1f}", ha="center", fontsize=7.5, fontweight="bold")

ax.set_title("Manual Quality Scores by System  (fp16, scale 0–5)",
             fontsize=13, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(SCORE_LABELS, fontsize=11)
ax.set_ylabel("Average Score (0–5)", fontsize=10)
ax.set_ylim(0, 5.5)
ax.legend(fontsize=10)
ax.spines[["top","right"]].set_visible(False)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
out = DOCS / "quality_scores_fp16.png"
plt.savefig(out, bbox_inches="tight")
plt.show()
print(f"Saved: {out}")

## Cell 5 — fp16 vs AWQ Quality Comparison (RAG Baseline)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

x     = np.arange(len(SCORE_LABELS))
width = 0.35

for i, (rows, backend, color) in enumerate([
    (rows_fp16, "fp16 (pytorch)",    "#4C72B0"),
    (rows_awq,  "AWQ (turbomind)",   "#DD8452"),
]):
    scores = avg_scores(rows, "rag_baseline")
    vals   = [scores[c] for c in SCORE_COLS]
    bars   = ax.bar(x + (i-0.5)*width, vals, width, label=backend,
                    color=color, edgecolor="white", alpha=0.9)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.04,
                f"{v:.2f}", ha="center", fontsize=8.5, fontweight="bold")

ax.set_title("RAG Baseline: fp16 vs AWQ Quality Scores  (scale 0–5)",
             fontsize=13, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(SCORE_LABELS, fontsize=11)
ax.set_ylabel("Average Score (0–5)", fontsize=10)
ax.set_ylim(0, 5.5)
ax.legend(fontsize=10)
ax.spines[["top","right"]].set_visible(False)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
out = DOCS / "quality_fp16_vs_awq.png"
plt.savefig(out, bbox_inches="tight")
plt.show()
print(f"Saved: {out}")

## Cell 6 — Radar Chart: System Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5),
                          subplot_kw=dict(polar=True))
fig.suptitle("Quality Scores Radar — All Systems",
             fontsize=13, fontweight="bold", y=1.03)

N      = len(SCORE_LABELS)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles += angles[:1]

for ax, (rows, title) in zip(axes, [
    (rows_fp16, "fp16 (pytorch)"),
    (rows_awq,  "AWQ (turbomind)"),
]):
    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(SCORE_LABELS, fontsize=9)
    ax.set_ylim(0, 5)
    ax.set_yticks([1, 2, 3, 4, 5])
    ax.set_yticklabels(["1","2","3","4","5"], fontsize=7, color="grey")
    ax.set_title(title, fontsize=11, fontweight="bold", pad=15)

    for sys, label, color in zip(SYSTEMS, SYS_LABELS, COLORS):
        scores = avg_scores(rows, sys)
        vals   = [scores[c] for c in SCORE_COLS]
        vals  += vals[:1]
        ax.plot(angles, vals, color=color, linewidth=2, label=label)
        ax.fill(angles, vals, color=color, alpha=0.1)

    ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.15), fontsize=8)

plt.tight_layout()
out = DOCS / "quality_radar.png"
plt.savefig(out, bbox_inches="tight")
plt.show()
print(f"Saved: {out}")

## Cell 7 — Summary Tables

In [ ]:
print("=" * 65)
print("DEPLOYMENT PERFORMANCE SUMMARY  (Colab A100 80GB)")
print("=" * 65)
hdr = f"{'Backend':<20} {'TTFT(ms)':>10} {'TPS':>8} {'Avg(s)':>8} {'VRAM(GB)':>10}"
print(hdr)
print("-" * len(hdr))
for eid, label in [("lmdeploy_fp16","fp16 (pytorch)"),("lmdeploy_awq","AWQ (turbomind)")]:
    e = experiments[eid]
    print(f"{label:<20} {e['ttft_ms']:>10} {e['estimated_tps']:>8} "
          f"{e['avg_response_s']:>8} {e['vram_peak_gb']:>10}")

print()
print("AWQ speedup:")
fp = experiments["lmdeploy_fp16"]
aw = experiments["lmdeploy_awq"]
print(f"  TTFT    : {fp['ttft_ms']/aw['ttft_ms']:.1f}x faster")
print(f"  TPS     : {aw['estimated_tps']/fp['estimated_tps']:.2f}x higher")
print(f"  Avg resp: {fp['avg_response_s']/aw['avg_response_s']:.2f}x faster")

print()
print("=" * 65)
print("QUALITY SCORES SUMMARY  (fp16 backend, scale 0–5)")
print("=" * 65)
hdr2 = f"{'System':<16} {'Correct':>8} {'Evidence':>9} {'Ground':>8} {'Tool':>6} {'Effic':>7} {'TOTAL':>7}"
print(hdr2)
print("-" * len(hdr2))
for sys, label in zip(SYSTEMS, SYS_LABELS):
    s = avg_scores(rows_fp16, sys)
    total = sum(s.values())
    print(f"{label:<16} {s['score_correctness']:>8.2f} {s['score_evidence']:>9.2f} "
          f"{s['score_groundedness']:>8.2f} {s['score_tool_use']:>6.2f} "
          f"{s['score_efficiency']:>7.2f} {total:>7.2f}")

print()
print("=" * 65)
print("fp16 vs AWQ QUALITY COMPARISON  (rag_baseline)")
print("=" * 65)
print(hdr2)
print("-" * len(hdr2))
for rows, backend in [(rows_fp16,"fp16"),(rows_awq,"AWQ")]:
    s = avg_scores(rows, "rag_baseline")
    total = sum(s.values())
    print(f"{backend:<16} {s['score_correctness']:>8.2f} {s['score_evidence']:>9.2f} "
          f"{s['score_groundedness']:>8.2f} {s['score_tool_use']:>6.2f} "
          f"{s['score_efficiency']:>7.2f} {total:>7.2f}")

print()
print("Charts saved to docs/:")
for f in sorted(DOCS.glob("*.png")):
    print(f"  {f.name}")